# Configure Projector Dimensions
In this notebook, we are going to use Dask to distribute finding the best hyper parameter configuration for the projector at the configuration dimensions. This is a modification of the `2026-03-06-DM-configure-projector.ipynb` notebook but does it all in Dask and completes to proecss by checking over the dimensions, and loads from the configuration file.

In [1]:
from hproj.data.paths import Paths

# these should be loaded from teh config file, but for now we hardcode them here
datasets = ['kather100k']
encoders = ['uni2']

# load in the datasets
paths = Paths.from_env()
k100k_uni2 = paths.embedding('kather100k', 'uni2')
k100k_uni2_train, k100k_uni2_test = k100k_uni2.load_splits()

# stratified downsample
# ten_percent = k100k_uni2_train.num_samples() * 0.1
# per_class = int(ten_percent / k100k_uni2_train.num_classes())
# sampled_k100k_uni2_train = k100k_uni2_train.stratified_sample(per_class)

ten_percent = k100k_uni2_train.num_samples() * 0.1
sampled_k100k_uni2_train = k100k_uni2_train.stratified_sample(ten_percent)
sampled_k100k_uni2_train.describe()

{'features': '(9999, 1536), float32',
 'labels': '(9999,), int64',
 'metrics': {}}

In [2]:
# bring in the scoring functions
from statistics import mean

import numpy as np

from hproj.data.feature_space import FeatureSpace
from hproj.data.folds import Fold, generate_stratified_folds
from hproj.measure.measurement import Measurement, MeasurementFactory
from hproj.projectors.projector import Projector, ProjectorFactory
from hproj.util.config import MeasurementConfig


def score_projector_config(embeddings: FeatureSpace, projector: Projector, measurements: list[Measurement], folds: list[Fold]) -> dict:
    # this function scores a projector (that has been configured with some hyperparameters) 
    # by applying it to the given embeddings and measuring the quality of the resulting projection with the given measurement. 
    # The score is averaged across the given folds.
    fold_scores = {measurement.name: [] for measurement in measurements}
    for train, valid in embeddings.get_folds(folds):
        projector.fit(train)
        train_proj = projector.transform(train)
        valid_proj = projector.transform(valid)
        for measurement in measurements:
            key = measurement.name
            score = measurement(train_proj, valid_proj, train, valid)
            fold_scores[key].append(score)

    # compute the mean for each measurement across the folds
    mean_scores = {key: float(mean(fold_scores)) for key, fold_scores in fold_scores.items()} 
    return mean_scores


# we need a function that wraps score_projector_config into a task that can be parallelized with different seeds
# this is done over all the hyperparameter configurations, for each dimension, for each seed, and for each projector type
# this function runs in the Dask task and is responsible for constructing the classes
# each one also generates a random seed for the projector using the calibration seed and the dimension and the projector type, 
# so that the same seed is used for the same configuration across different runs
# note that we are also doing this so that we don't accidently share state
# from hproj.util.config import ProjectorConfig

def score_projector_config_task(embeddings_future, dataset_name: str, proj_name: str, proj_hyperparams: dict, measurement_configs: list[MeasurementConfig], n_components, seeds, folds):
    def score(seed):
        projector = ProjectorFactory.create(proj_name, n_components, seed, **proj_hyperparams)
        measurements = [MeasurementFactory.create(c.name, seed, **c.params) for c in measurement_configs]
        scores = score_projector_config(embeddings_future, projector, measurements, folds)
        return scores

    def summarize(results):
        keys = results[0].keys()
        summary = {}

        for k in keys:
            values = np.array([r[k] for r in results])
            summary[f"{k}-mean"] = values.mean()
            summary[f"{k}-std"] = values.std(ddof=1)  # sample std

        return summary

    # compute the scores for the measurements over all the seeds
    scores_for_seeds = [score(seed) for seed in seeds]

    # report mean ± std over N runs
    results = summarize(scores_for_seeds)

    # add the meta data
    meta_data = {
        'projector': proj_name,
        'dataset': dataset_name, 
        'n_components': n_components,
        'num_seeds': len(seeds)
    }
    return meta_data | proj_hyperparams | results


In [3]:
# load in the config file
from hproj.util.config import Config

cfg = Config.from_yaml('../configs/basic.yml')
cfg

Config(datasets=['kather100k', 'spider-colorectal', 'spider-breast', 'spider-skin', 'spider-thorax'], encoders=['uni', 'uni2', 'hibou', 'dinov2'], seeds=SeedConfig(calibration=[0, 1], curve=[0, 1, 2], evaluation=[0, 1, 2, 3, 4]), num_folds=5, calibration=CalibrationConfig(dimensions=[1, 2], measurements=[MeasurementConfig(name='mean-knn-score', params={'ks': [5, 15, 30]})], projectors=[ProjectorConfig(name='pca', params={}), ProjectorConfig(name='grp', params={}), ProjectorConfig(name='umap', params={'n_neighbors': [2, 5], 'min_dist': [0.0, 0.1, 0.99], 'metric': ['cosine']})], subsample=10000))

### From the README:
For each of projectors generate a set of hyperparameter configurations by the grid.
At each of the *callibration dimensions* and for each of the projector, for each hyperparameter configuration, for each fold of the data:
1. fit on the train of the fold
2. transform the train and valid of the fold
3. return the k-nn accuracy for the valid. Note - The k-nn *score* is computed of *values of k* and averaged.
The mean of the folds to find the k-nn score cross validation value for that configuration at that dimension.

Select configuration that performs best across all dimensions and across all datasets (take the mean score over all dimensions for each configurations).

Repeat this *n-times* for stochastic projectors and take the best of the n runs. This is using paired seeds, so have a set of seed defined upfront and reuse them multiple times.

In [ ]:
from time import perf_counter

import pandas as pd
from dask.distributed import Client
from dask_cuda import LocalCUDACluster

from hproj.util.hyperparams import make_param_grid

datasets = {'sampled_k100k_uni2_train': sampled_k100k_uni2_train}


# get the number of dims from the config file
dims = cfg.calibration.dimensions

# get the calibration seeds from the config file
seeds = cfg.seeds.calibration

num_folds = cfg.num_folds

# note that the folds should be generated from the sampled training set, not the full training set
# they are generated once using a fixed seed, the paired seeds are used to the projectors

dataset_folds = {
    name: generate_stratified_folds(ds, n_splits=num_folds, seed=42)
    for name, ds in datasets.items()
}

# set up the dask client
def make_dask_client():
    cluster = LocalCUDACluster(
        threads_per_worker=1,
    )
    client = Client(cluster)
    return client, cluster
client, cluster = make_dask_client()

# timings
start_time = perf_counter()

# for each projector in the calibration config
results = {}
for projector in cfg.calibration.projectors:
    print(f"Evaluating projector: {projector.name}: hyperparameters {projector.params}")
    if len(projector.params) == 0:
        print("No hyperparameters to tune, skipping.")
        continue
    
    # geneate the hyperparameter grid for this projector
    param_grid = make_param_grid(projector.params)
    print(f'Generated {len(param_grid)} hyperparameter combinations to evaluate.')

    # we want the calibration for each projector over all datasets
    projector_results = []
    for dataset_name, embeddings in datasets.items():
        print(f"Evaluating projectors ondataset: {dataset_name}")
        
        embeddings_future = client.scatter(embeddings, broadcast=False)
        folds = dataset_folds[dataset_name]

        for n_components in dims:
            futures = [
                client.submit(
                    score_projector_config_task,
                    embeddings_future, 
                    dataset_name, 
                    projector.name,
                    projector_hyperparams, 
                    cfg.calibration.measurements, 
                    n_components, 
                    seeds, 
                    folds,
                    pure=False,
                )
                for projector_hyperparams in param_grid
            ]

            projector_results.extend(client.gather(futures))
                
    results['projector'] = pd.DataFrame(projector_results)

# sort based on the knn score
for projector_name, results_df in results.items():
    print(projector_name)
    df_sorted = results_df.sort_values(["dataset", "mean-knn-score-mean"], ascending=False)
    print(df_sorted)
    
elapsed_time = perf_counter() - start_time
print(f'\nTotal time: {elapsed_time:.2f} seconds')
    

Evaluating projector: pca: hyperparameters {}
No hyperparameters to tune, skipping.
Evaluating projector: grp: hyperparameters {}
No hyperparameters to tune, skipping.
Evaluating projector: umap: hyperparameters {'n_neighbors': [2, 5], 'min_dist': [0.0, 0.1, 0.99], 'metric': ['cosine']}
Generated 6 hyperparameter combinations to evaluate.
Evaluating projectors ondataset: sampled_k100k_uni2_train
projector
   projector                   dataset  n_components  num_seeds  n_neighbors  \
10      umap  sampled_k100k_uni2_train             2          2            5   
9       umap  sampled_k100k_uni2_train             2          2            5   
11      umap  sampled_k100k_uni2_train             2          2            5   
3       umap  sampled_k100k_uni2_train             1          2            5   
4       umap  sampled_k100k_uni2_train             1          2            5   
5       umap  sampled_k100k_uni2_train             1          2            5   
7       umap  sampled_k100k_uni